# Dự án: Phát hiện đám cháy bằng YOLOv8 (Phat_hien_dam_chay_Yolov8)

Notebook này được chuẩn bị để huấn luyện mô hình YOLOv8 phát hiện đám cháy và khói trên Google Colab (sử dụng GPU T4 miễn phí).

### 1. Kiểm tra cấu hình GPU

In [1]:
!nvidia-smi

Wed Oct  7 11:23:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P5             16W /   75W |     350MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Cài đặt thư viện Ultralytics (YOLOv8)

In [2]:
!pip install ultralytics roboflow


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 3. Kết nối Google Drive (Tùy chọn nếu lưu dataset trên Drive)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

### 4. Giải nén Dataset
> **Lưu ý:** Upload file `Phat_hien_dam_chay_Yolov8.zip` lên Google Colab hoặc Google Drive, sau đó chạy lệnh giải nén dưới đây.

In [ ]:
# Nếu upload trực tiếp lên Colab:
!unzip -q /content/Phat_hien_dam_chay_Yolov8.zip -d /content/dataset

# Hoặc nếu lấy từ Google Drive:
# !unzip -q '/content/drive/MyDrive/Phat_hien_dam_chay_Yolov8.zip' -d /content/dataset

### 5. Cấu hình lại đường dẫn file `data.yaml`

In [ ]:
yaml_content = """
train: /content/dataset/train/images
val: /content/dataset/valid/images
test: /content/dataset/test/images

nc: 2
names: ['Fire', 'Smoke']
"""

with open('/content/dataset/data.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())

print("Cập nhật data.yaml thành công!")

### 6. Huấn luyện mô hình YOLOv8 (Training)

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình pretrained yolov8s (hoặc yolov8n nếu muốn nhẹ hơn)
model = YOLO('yolov8s.pt')

# Bắt đầu huấn luyện
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='Phat_hien_dam_chay_Yolov8',
    name='train_run',
    save=True
)

### 7. Đánh giá mô hình trên tập Test

In [ ]:
# Đánh giá độ chính xác (mAP, Precision, Recall)
metrics = model.val(data='/content/dataset/data.yaml', split='test')

### 8. Dự đoán thử nghiệm (Inference)

In [ ]:
import glob
test_images = glob.glob('/content/dataset/test/images/*.jpg')[:5]
results = model.predict(source=test_images, save=True, conf=0.25)

### 9. Nén kết quả và lưu về Google Drive hoặc tải xuống

In [ ]:
# Nén toàn bộ trọng số (best.pt) và kết quả train
!zip -r /content/Phat_hien_dam_chay_results.zip /content/Phat_hien_dam_chay_Yolov8

# Lưu sang Drive (nếu đã mount)
# !cp /content/Phat_hien_dam_chay_results.zip /content/drive/MyDrive/